# Module 08 — Attention, the Idea Behind Transformers

The bigram model only saw the **last** word. Real LLMs consider **all** prior
words and decide which ones matter for predicting the next. That mechanism is
**attention**, the heart of the "Transformer" architecture behind GPT and Claude.

Attention answers: *"To understand this word, which other words should I look at,
and how much?"* We'll compute it by hand on a 4-word sentence.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

# 4 words, each already turned into a 3-dim embedding (Module 08 embeddings).
words = ["the", "cat", "sat", "mat"]
E = np.array([
    [1.0, 0.0, 0.2],   # the
    [0.9, 0.8, 0.1],   # cat
    [0.2, 0.9, 0.7],   # sat
    [0.8, 0.7, 0.9],   # mat
])

## Query, Key, Value
Each word produces three vectors (via learned weight matrices — here random):
- **Query (Q):** "what am I looking for?"
- **Key (K):** "what do I offer?"
- **Value (V):** "what do I contribute if attended to?"

A word's attention to another = how well its Query matches the other's Key.

In [ ]:
rng = np.random.default_rng(0)
d = 3
Wq, Wk, Wv = rng.normal(0, 1, (d, d)), rng.normal(0, 1, (d, d)), rng.normal(0, 1, (d, d))
Q, K, V = E @ Wq, E @ Wk, E @ Wv
print("Q shape", Q.shape, "= one query vector per word")

## Step 1 — attention scores = Q · Kᵀ
Every word scores every other word by dot-product similarity (Module 02 again!).

In [ ]:
scores = Q @ K.T / np.sqrt(d)     # scaled dot-product
print("raw attention scores (row = word doing the looking):")
for i, w in enumerate(words):
    print(f"  {w:4s}:", scores[i])

## Step 2 — softmax turns scores into weights that sum to 1
Now each word has a distribution over all words: "how much of my attention goes
to each."

In [ ]:
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

weights = softmax(scores)
print("attention weights (each row sums to 1):")
for i, w in enumerate(words):
    row = ", ".join(f"{words[j]}={weights[i,j]:.2f}" for j in range(len(words)))
    print(f"  {w:4s} attends to -> {row}")

## Step 3 — output = weighted sum of Values
Each word's new representation blends the Values of the words it attended to.
That blended vector is what flows to the next layer.

In [ ]:
output = weights @ V
print("new context-aware representation per word:")
for i, w in enumerate(words):
    print(f"  {w:4s}:", output[i])

## Why this is a big deal
- Every word can pull information from **any** other word in the sequence, no
  matter how far away — solving the bigram's "only sees the last word" problem.
- The weights are **data-dependent**: "it" can learn to attend to whichever noun
  it refers to. This is how models track long-range meaning.
- Stack many attention layers + feed-forward layers = a **Transformer**. Add
  billions of parameters and trillions of training tokens = an LLM.

### Security note
Attention operates over the *entire* context, including any text a user pastes in.
That's the mechanistic root of **prompt injection**: malicious instructions hidden
in retrieved/pasted content get attended to just like trusted instructions. You'll
tackle that defense directly in Module 10.

**Next:** back to `tutorial.html`, then `project.md`.